### Method
Using agentic workflow to categorize bank transactions


1. use monoply to process the pdfs into csv
2. merge csvs into one
3. use llm to work parallely to categorize our transactions [only to categorize] with reviewer LLM to check if input length is correct.
4. if unknown then we use a researcher agent -> We clean using regex then duck duck go search to gather context for unknown lines + RAG and call LLM.
4. manually tableau later

In [229]:
import os
import subprocess

# Processes our ~pdfs
def process_pdf():
    subprocess.run(["monopoly", ".", '--preserve-filename'])
    return 'Finished Processing PDFs'


In [430]:
process_pdf()

'Finished Processing PDFs'

### Read CSVs

In [509]:
import pandas as pd

df = pd.read_csv('test2.csv')

### DuckDuck Go SCRAPE 

In [315]:
import requests
from bs4 import BeautifulSoup

def ddg_scrape(query, max_results=3):
    url = "https://duckduckgo.com/html/"
    params = {
        "q": query,
        "kl": "sg-en"
    }
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    res = requests.get(url, params=params, headers=headers)
    soup = BeautifulSoup(res.text, "html.parser") #[parser]

    results = []

    for result in soup.select(".result"):
        title_tag = result.select_one(".result__a")
        snippet_tag = result.select_one(".result__snippet")

        if title_tag:
            title = title_tag.get_text()
            snippet = snippet_tag.get_text() if snippet_tag else ""
            results.append({
                "title": title,

                "snippet": snippet
            })
        if len(results) >= max_results:
            break

    return results

### Cleaning

In [233]:
import regex as re

In [300]:
def clean_line(line):
    line = line.replace("FAST PAYMENT via PayNow-Mobile", "")
    line = line.replace("FAST PAYMENT via PayNow-QR", "")
    line = line.replace("FAST PAYMENT via PayNow-UEN to ", "")
    line = line.replace("PAYMENT/TRANSFER DBSS from ", "")  
    line = line.replace("PAYMENT/TRANSFER OCBC from ", "")  
    line = line.replace("PAYMENT/TRANSFER UOB from ", "")  
    line = line.replace("PAYMENT/TRANSFER HSBC from ", "")  
    line = line.replace("PAYMENT/TRANSFER MAYBANK from ", "")
    line = line.replace("PAYMENT/TRANSFER CIMB from ", "")
    line = line.replace("PAYMENT/TRANSFER STANDARD CHARTERED from ", "")
    line = line.replace("PAYMENT/TRANSFER DBS from ", "")
    line = line.replace("PAYMENT/TRANSFER POSB from ", "")
    line =  re.sub(r'^.*?NETS QR PURCHASE\s+', '', line)
    line = re.sub(r'DEBIT PURCHASE \d{2}/\d{2}/\d{2} [xX]{2}-\d{4}', '', line)
    line = re.sub(r'CARD PURCHASE \d{2}/\d{2}/\d{2} [xX]{2}-\d{4}', '', line)
    line = re.sub(r'CREDIT PURCHASE \d{2}/\d{2}/\d{2} [xX]{2}-\d{4}', '', line)
    line = re.sub(r'OTHR-.*$', '', line)
    return line.strip()


In [301]:
clean_line("NETS QR 91641001 NETS QR PURCHASE NUS U TOWN ST02")

'NUS U TOWN ST02'

### LLM Calls

In [ ]:
from dotenv import load_dotenv

In [232]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

assert GROQ_API_KEY is not None, "GROQ_API_KEY environment variable not set"

In [ ]:
def check_output_length(original_input, ai_output):
    if len(original_input) != len(ai_output.split(',')):
        return "Incorrect"
    return "Correct"

In [475]:
from pydantic_ai import Agent
from pydantic_ai.models.groq import GroqModel
from pydantic_ai.providers.groq import GroqProvider

model = GroqModel(
    'openai/gpt-oss-120b', provider=GroqProvider(api_key=GROQ_API_KEY) #Smaller Model
)
agent = Agent(
    model,
    system_prompt=(
        "You are a Singapore-based bank transaction classifier. "
        "Task: Categorize each transaction into: Food, Utilities, Transport, Healthcare, Shopping, Travel, Other, Uncertain. "

        "Classification Rules (Priority Order): "
        "1. Transport: GRAB (non-food), GOJEK, COMFORTDELGRO, TRANSITLINK, SGBIKE, SIMPLYGO, EZ-LINK. "
        "2. Food: Restaurants, cafes, food courts, fast food(McDonalds,KFC,Stuff'd,Dabba Street), delivery (GrabFood, FoodPanda), and merchants with food-related keywords like 'Eat', 'F&B', 'CAFE', 'EATERY', or 'COFFEE'."
        "3. Utilities: Telcos (Singtel, StarHub, M1, GOMO, SIMBA, VIVIFI), SP Group, and recurring bills. "
        "4. Healthcare: Clinics, hospitals, and pharmacies (e.g., Guardian, Watsons). "
        "5. Shopping: Supermarkets (NTUC, Cold Storage, Sheng Siong, Donki), e-commerce (Shopee, Lazada, Amazon), retail stores, fitness merchants (Anytime Fitness) and entertainment (KTV, Skating, Arcade) "
        "6. Travel: Airlines, hotels, travel agencies and online booking platforms (e.g., Expedia, Booking.com, Trip, Agoda). "
        "7. Other: This is for financial movements and personal transfers. Includes: "
        "   - Digital wallet top-ups (e.g., REVOLUT, YOUTRIP, WISE, GRABPAY). "
        "   - P2P transfers and Fund Transfers (PAYNOW, PAYLAH) to individual names. "
        "   - Bank fees, interest, or internal transfers. "
        "8. Uncertain: Use this for ambiguous codes, generic UENs, or any merchant where confidence is <95%. "

        "Output Format: "
        "- Strictly output ONLY the category names. "
        "- Maintain the input order. "
        "- Separate by commas (no spaces). "
        "- DO NOT include explanations or extra text. "

        "Example: "
        "Input: GRAB* 1234|REVOLUT TOPUP|NTUC FAIRPRICE|PAYNOW TO DAN "
        "Output: Transport,Other,Shopping,Other"
    )
)

In [464]:
# Reviewer model
reviewer_model = GroqModel(
    'openai/gpt-oss-120b', provider=GroqProvider(api_key=GROQ_API_KEY)
)

# Tend to hallucinate in the end, so we keep the initial values but ensure it checks the last few. # Biger Model
reviewer_agent = Agent(
    reviewer_model,
   system_prompt=(
        "You are a Singapore-based Data Integrity Auditor. "
        "Your sole mission is to ensure a 1:1 mapping between transactions and categories. "
        "These are the categories: Food, Utilities, Transport, Healthcare, Shopping, Travel, Other, Uncertain."
        "REFERENCE CLASSIFICATION STANDARDS (Priority Order):"
        "1. Transport: All ride-hailing (GRAB, GOJEK) and public transit (SIMPLYGO, EZ-LINK)."
        "2. Food: All dining, cafes, and delivery. Key triggers: 'BREW', 'CAFE', 'EATERY', 'COFFEE', 'GrabFood'."
        "3. Utilities: All SG Telcos (Singtel, StarHub, M1, GOMO, SIMBA) and SP Group."
        "4. Healthcare: Medical clinics, hospitals, and pharmacies (Guardian/Watsons)."
        "5. Shopping: Supermarkets (NTUC, Donki), E-commerce (Shopee/Lazada), Fitness (Anytime Fitness), and Entertainment (KTV/Arcade)."
        "6. Travel: Airlines, Hotels, and Booking sites (Expedia/Agoda)."
        "7. Other: Wallet top-ups (REVOLUT, YOUTRIP), P2P (PayNow/PayLah), and Bank Fees."
        "8. Uncertain: Only use if the merchant name is a generic UEN or completely unrecognizable."

        "Input Analysis: "
        "You will receive a list formatted as: 'Transaction Description' -> 'Attempted Category'. "
        
        "RECONCILIATION RULES: "
        "1. ALIGNMENT: Every transaction MUST have exactly one category. "
        "2. CORRECTION: If 'Attempted Category' is [MISSING] or [INCORRECT], assign the correct category based on Singapore merchant rules. "
        "3. TRIMMING: If there are extra categories provided that do not have a corresponding transaction, DISCARD them immediately. "
        "4. PRESERVATION: If the 'Attempted Category' is already correct, do not change it. "

        "STRICT OUTPUT: "
        "- Output ONLY the final category names. "
        "- Format: Comma-separated, no spaces, no trailing commas. "
        "- The number of categories in your output MUST exactly match the number of input transactions."
    )
)

def correction_prompt(original_input, ai_output):
    return (
        f"The original input had {len(original_input)} transactions, but the AI output {len(ai_output)} categories. Please correct the output to have exactly {len(original_input)} categories, strictly following the rules and format. Only output the corrected comma-separated categories without any explanations or extra text."
        f"These are the original transactions: {original_input} and the AI's assigned categories: {ai_output}."
    )

In [485]:
# Research Agent 

# Research model
researcher_agent = GroqModel(
    'meta-llama/llama-4-scout-17b-16e-instruct', provider=GroqProvider(api_key=GROQ_API_KEY)
)

researcher_agent = Agent(
    reviewer_model,
    system_prompt=(
        "You are a Merchant Intelligence Researcher. "
        "Your task: Analyze scraped metadata to identify a merchant's primary business. "
        
        "RULES: "
        "1. Identify the core 'Category' from: Food, Utilities, Transport, Healthcare, Shopping, Travel, Other."
        "2. If the data mentions food, beverages, dining, or cafes (e.g., 'Brew', 'Roast', 'Rice'), prioritize **Food**. "
        "3. If the data is ambiguous, output **Uncertain**."
    
        "Output Format: Strictly the Category name."
    )
)

def research_prompt(info):
    return (
        f"{info}"
    )

In [510]:
async def run_agent(agent,li: str):
    result = await agent.run(li)
    return result.output

In [432]:
df2 = df[0:20]

In [511]:
async def classify_transactions(transactions): #df['description]
    c = 10 # chunk size
    all = []
    for i in range(0, len(transactions), c):
        trans_sliced = transactions[i:i+c]
        li = ' | '.join(trans_sliced.tolist()) # creates list
        try:
            output = await run_agent(agent,li)
            cleaned = [x.strip() for x in output.split(',')]
            if check_output_length(trans_sliced,output) == 'Incorrect':
                print("Output length incorrect, running reviewer agent...")
                reviewed =  await run_agent(reviewer_agent, correction_prompt(trans_sliced.to_list() , cleaned))
                reviewed_list = [x.strip() for x in reviewed.split(',')]
                #print('Agent-Reviewed Output:', reviewed)
                #print("Original:" , cleaned)
                all.extend(reviewed_list)
            else:
                all.extend(cleaned)
                print('Non Agent-Reviewed Output:', cleaned)
        except Exception as e:
            print('Error', e)
    return all


In [512]:
a = await classify_transactions(df['description'])

Output length incorrect, running reviewer agent...
Non Agent-Reviewed Output: ['Food', 'Other', 'Food', 'Other', 'Food', 'Food', 'Other', 'Other', 'Other', 'Other']
Output length incorrect, running reviewer agent...
Non Agent-Reviewed Output: ['Other', 'Food', 'Food', 'Other', 'Transport', 'Food', 'Other', 'Uncertain', 'Uncertain', 'Uncertain']
Non Agent-Reviewed Output: ['Travel', 'Other', 'Other', 'Other', 'Other', 'Other', 'Other', 'Other', 'Other', 'Other']
Non Agent-Reviewed Output: ['Other', 'Other', 'Other', 'Other', 'Food', 'Food', 'Other', 'Other', 'Other', 'Other']
Non Agent-Reviewed Output: ['Other', 'Other', 'Other', 'Other', 'Utilities', 'Other', 'Transport', 'Other', 'Other', 'Other']
Non Agent-Reviewed Output: ['Food', 'Food', 'Food', 'Food', 'Other', 'Other']


In [515]:
df['Tag'] = a

In [516]:
df

,date,description,amount,balance,Tag
0,2026-01-01,DEBIT PURCHASE 30/12/25 xx-1491 MCDONALD'S (PR...,-10.60,336.31,Food
1,2026-01-01,FAST PAYMENT via PayNow-UEN to STRIPE PAYMENTS...,-2.99,333.32,Other
2,2026-01-01,CASH DEPOSIT CDM xx-1491 OCBC-WHITE SANDS S,10.00,343.32,Other
3,2026-01-01,CASH DEPOSIT CDM xx-1491 OCBC-WHITE SANDS S,67.00,410.32,Other
4,2026-01-01,FAST PAYMENT via PayNow-NRIC/FIN to TOLENTINO ...,-100.00,310.32,Other
...,...,...,...,...,...
71,2026-01-30,NETS QR 90839801 NETS QR PURCHASE BOTAK DELICACY,-5.40,133.59,Food
72,2026-01-30,NETS QR 91734701 NETS QR PURCHASE OTMH - DRINK...,-1.70,131.89,Food
73,2026-01-31,FAST PAYMENT via PayNow-Mobile to LUCKIN LUCKY...,-3.00,128.89,Food
74,2026-01-31,COLLECTION/TRANSFER OTHR REVOLUT TECHNO REVOLU...,-30.00,98.89,Other


In [517]:
df[df['Tag'] == 'Uncertain']

,date,description,amount,balance,Tag
37,2026-01-15,NETS QR 91641001 NETS QR PURCHASE NUS U TOWN ST02,-6.0,510.89,Uncertain
38,2026-01-15,NETS QR 68436501 NETS QR PURCHASE NUS U TOWN ST14,-1.5,509.39,Uncertain
39,2026-01-15,NETS QR 68436501 NETS QR PURCHASE NUS U TOWN ST14,-0.1,509.29,Uncertain


In [518]:
ddg_scrape(clean_line("NETS QR 91641001 NETS QR PURCHASE NUS U TOWN ST02"))

[{'title': 'UTown - NUS UCI',
  'snippet': 'Designed for the entire NUS community, University Town, or UTown® for short, is strategically integrated with the Kent Ridge Campus via a vehicle and pedestrian bridge.'},
 {'title': 'University Town, Nus - Info Opening Hours, Address and Latest Visitor ...',
  'snippet': 'University Town, Nus on 2 College Ave West, Stephen Riady Centre, Singapore 138607. Complete information for University Town, NUS, service opening hours, location address, and visitor reviews on University Town, NUS.'},
 {'title': 'Campus Services — UTown - NUS UCI',
  'snippet': 'Could you provide a contact number / email address in case I have more questions about UTown® (non-residential)? Please feel free to write to us at utown@nus.edu.sg. Alternatively, you may call our general enquiry line at 6601-2135.'}]

In [521]:
res3 = await researcher_agent.run(research_prompt(ddg_scrape(clean_line("NETS QR 91641001 NETS QR PURCHASE NUS U TOWN ST02"))))

In [522]:
res3.output

'Other'

In [472]:
len(a)

41

In [477]:
a

['Other',
 'Other',
 'Other',
 'Food',
 'Food',
 'Food',
 'Food',
 'Other',
 'Food',
 'Food',
 'Other',
 'Other',
 'Other',
 'Other',
 'Uncertain',
 'Other',
 'Other',
 'Shopping',
 'Food',
 'Other',
 'Food',
 'Other',
 'Food',
 'Food',
 'Other',
 'Food',
 'Shopping',
 'Food',
 'Other',
 'Other',
 'Other',
 'Other',
 'Other',
 'Other',
 'Utilities',
 'Other',
 'Other',
 'Other',
 'Other',
 'Other',
 'Other']

In [476]:
len(df)

41